# IoMT Compression — 01 Download CICIoMT2024 + Inventory

Downloads the **CICIoMT2024** CSV flow-feature files (resume-safe) and then runs the **metadata feasibility check** that decides the split design.

**The critical question this notebook answers on day one:** do the CSVs carry device identifiers and per-flow timestamps? Device-held-out and temporal split protocols depend on them. If absent, those two protocols need the PCAPs re-extracted, or get descoped. Do not skip the inventory section.

Source: Kaggle mirror `amineipad/cic-iomt-dataset-2024` (official: unb.ca/cic/datasets/iomt-dataset-2024.html). Needs a Kaggle API token (`kaggle.json`) on Drive root, same persistence pattern as the git PAT.

In [ ]:
# --- BOOTSTRAP: mount Drive, restore git creds, chdir to repo, import config ---
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, sys
DRIVE_ROOT = '/content/drive/MyDrive/IOMT_Compression_Research'
REPO = '/content/drive/MyDrive/IOMT_Compression_Research/iomt-compression-research'
for f in ['.gitconfig', '.git-credentials']:
    src = f'{DRIVE_ROOT}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)
        print(f'restored {f}')
    else:
        print(f'WARNING {f} not on Drive — run 00_setup first')
os.chdir(REPO)
sys.path.append(os.path.join(REPO, 'src'))
from iomtc_config import PATHS, C, set_seed, ensure_dirs
ensure_dirs(); set_seed()
!git pull -q
print('ready in', os.getcwd())

In [ ]:
# --- 1. Kaggle auth: restore kaggle.json from Drive (or upload it once) ---
import os, shutil
os.makedirs('/root/.config/kaggle', exist_ok=True)
kaggle_drive = str(PATHS.kaggle_json)
kaggle_local = '/root/.config/kaggle/kaggle.json'
if os.path.exists(kaggle_drive):
    shutil.copy(kaggle_drive, kaggle_local)
    os.chmod(kaggle_local, 0o600)
    print('kaggle.json restored from Drive')
else:
    print('No kaggle.json on Drive. Upload it once:')
    from google.colab import files
    up = files.upload()   # choose your kaggle.json
    shutil.copy('kaggle.json', kaggle_drive)      # persist to Drive root
    shutil.copy('kaggle.json', kaggle_local)
    os.chmod(kaggle_local, 0o600)
    os.remove('kaggle.json')
    print('kaggle.json saved to Drive + installed')
!pip install -q kaggle

In [ ]:
# --- 2. Download CICIoMT2024 (resume-safe: skips if already present) ---
import os, subprocess
DEST = str(PATHS.cic_iomt2024)
os.makedirs(DEST, exist_ok=True)
SLUG = 'amineipad/cic-iomt-dataset-2024'

existing = [f for f in os.listdir(DEST) if f.lower().endswith(('.csv', '.zip'))]
if existing:
    print(f'Found {len(existing)} files already in {DEST} — skipping download.')
    print('Delete the folder contents to force a re-download.')
else:
    print(f'Downloading {SLUG} -> {DEST} (this is large; leave it running)')
    # --unzip extracts in place; Kaggle resumes partial downloads
    rc = subprocess.run(['kaggle','datasets','download','-d',SLUG,'-p',DEST,'--unzip'],
                        capture_output=True, text=True)
    print(rc.stdout[-2000:]); print(rc.stderr[-2000:])
    if rc.returncode != 0:
        print('\nKaggle download failed. Check: (a) kaggle.json valid, '
              '(b) dataset slug still live, (c) you accepted any dataset rules on the Kaggle page.')
        print('Official fallback: https://www.unb.ca/cic/datasets/iomt-dataset-2024.html')

In [ ]:
# --- 3. Inventory: what arrived (files, sizes, where the CSVs live) ---
import os
DEST = str(PATHS.cic_iomt2024)
csvs = []
for root, _, fns in os.walk(DEST):
    for fn in fns:
        if fn.lower().endswith('.csv'):
            fp = os.path.join(root, fn)
            csvs.append((os.path.relpath(fp, DEST), os.path.getsize(fp)))
csvs.sort()
print(f'{len(csvs)} CSV files under {DEST}\n')
tot = 0
for rel, sz in csvs[:60]:
    tot += sz
    print(f'  {sz/1e6:8.1f} MB  {rel}')
if len(csvs) > 60:
    print(f'  ... and {len(csvs)-60} more')
print(f'\nTotal CSV size: {sum(s for _,s in csvs)/1e9:.2f} GB')
# filename encodes the label in CICIoMT2024 — note the distinct stems
stems = sorted({os.path.basename(r).rsplit("_",1)[0] if "_" in os.path.basename(r) else os.path.basename(r) for r,_ in csvs})
print(f'\n{len(stems)} distinct filename stems (proxy for label families):')
for s in stems[:40]:
    print('  ', s)

In [ ]:
# --- 4. CRITICAL metadata feasibility check (gates the split design) ---
# Reads ONE csv's header + a small sample to see which columns exist.
import pandas as pd, os
DEST = str(PATHS.cic_iomt2024)
first_csv = None
for root, _, fns in os.walk(DEST):
    for fn in sorted(fns):
        if fn.lower().endswith('.csv'):
            first_csv = os.path.join(root, fn); break
    if first_csv: break
assert first_csv, 'no CSV found — run the download cell'
print('inspecting:', os.path.relpath(first_csv, DEST), '\n')
sample = pd.read_csv(first_csv, nrows=2000)
cols = [c.lower() for c in sample.columns]
print(f'{len(sample.columns)} columns:')
print(list(sample.columns), '\n')

def has(*keys):
    return [c for c in sample.columns if any(k in c.lower() for k in keys)]

device_cols = has('mac','device','src_ip','dst_ip','ip','addr','host')
time_cols   = has('time','timestamp','ts','date','flow_start','duration')
proto_cols  = has('protocol','proto','port','service')

print('--- SPLIT-PROTOCOL FEASIBILITY ---')
print('device-identifying cols :', device_cols or 'NONE  -> device-held-out NOT feasible from CSV')
print('timestamp/temporal cols :', time_cols   or 'NONE  -> temporal split NOT feasible from CSV')
print('protocol/port cols      :', proto_cols  or 'NONE  -> cross-protocol relies on filename/dir grouping')
print('\nIf device/timestamp are NONE: only random + cross-protocol (via file grouping) '
      'are possible from CSVs. Device-held-out and temporal need PCAP re-extraction — '
      'record this decision in data_version.md before Phase 1.')

In [ ]:
# --- 5. Class distribution from filename stems (rare-class check) + save report ---
# Counts rows per file stem WITHOUT loading everything into RAM at once.
import pandas as pd, os, csv as _csv
DEST = str(PATHS.cic_iomt2024)
rows_per_stem = {}
for root, _, fns in os.walk(DEST):
    for fn in sorted(fns):
        if not fn.lower().endswith('.csv'):
            continue
        fp = os.path.join(root, fn)
        # fast row count (minus header)
        with open(fp, 'r', errors='ignore') as f:
            n = sum(1 for _ in f) - 1
        stem = fn.rsplit('.',1)[0]
        rows_per_stem[stem] = rows_per_stem.get(stem, 0) + max(n, 0)
dist = sorted(rows_per_stem.items(), key=lambda kv: -kv[1])
total = sum(v for _,v in dist)
print(f'total rows across files: {total:,}\n')
for stem, n in dist[:40]:
    print(f'  {n:12,}  ({100*n/total:5.2f}%)  {stem}')

# persist to reports/ (tracked in git)
rep = os.path.join(str(PATHS.reports), 'cic_iomt2024_file_row_counts.csv')
os.makedirs(str(PATHS.reports), exist_ok=True)
with open(rep, 'w', newline='') as f:
    w = _csv.writer(f); w.writerow(['file_stem','rows','pct'])
    for stem, n in dist:
        w.writerow([stem, n, round(100*n/total,4)])
print('\nsaved', rep)

In [ ]:
# --- 6. Commit the inventory report (datasets stay on Drive, gitignored) ---
import os
os.chdir(REPO)
!git add reports/cic_iomt2024_file_row_counts.csv
!git commit -q -m "NB01: CICIoMT2024 downloaded; inventory + metadata feasibility report" || echo "nothing to commit"
!git push -q origin main
print('\ninventory committed. Datasets remain on Drive (gitignored).')
print('Drive saved + git pushed? Confirm before we move to Phase 1 splitting.')